# 🏥 CNN-Based ICD-10 Auto-Coding System
## Notebook 1: Environment Setup & Data Extraction

---

**Project Goal**: Build a CNN model to predict ICD-10 codes from medical documents

**This Notebook Covers**:
1. Environment Setup (mount Drive, install dependencies)
2. PDF Text Extraction with OCR fallback
3. ICD-10 Code Extraction and Validation
4. Dataset Analysis and Statistics

**Data**:
- 831 PDFs (346 PT/OT + 485 Home Health)
- ~60% need OCR, 40% text-selectable
- All contain ICD-10 codes

**Estimated Runtime**: ~3-4 hours for full extraction

---

## 📋 Section 1: Environment Setup

### 1.1 Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Drive mounted successfully!")

### 1.2 Install Required Libraries

In [ ]:
%%capture
# Install Python packages
!pip install pdfplumber PyPDF2 pdf2image pytesseract
!pip install nltk gensim
!pip install tqdm

# Install system dependencies for OCR
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr
!apt-get install -y -qq libtesseract-dev
!apt-get install -y -qq poppler-utils

print("✓ All packages installed!")

In [ ]:
# Verify installations
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import os
import re
import gc
import time
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple, Optional

# Check Tesseract
!tesseract --version | head -1

print("\n✓ All imports successful!")

### 1.3 Configure Paths

In [ ]:
# ============================================
# CONFIGURATION - EDIT THESE PATHS IF NEEDED
# ============================================

# Google Drive base
DRIVE_BASE = "/content/drive/MyDrive"

# Your PDF folders (VERIFY THESE PATHS!)
HOME_HEALTH_PATH = f"{DRIVE_BASE}/485/485 DOCS"
PT_OT_PATH = f"{DRIVE_BASE}/485/Other Than 485"

# Project output folder
PROJECT_FOLDER = f"{DRIVE_BASE}/ICD10_Project"
DATA_FOLDER = f"{PROJECT_FOLDER}/data"
PROCESSED_FOLDER = f"{DATA_FOLDER}/processed"

# Output files
OUTPUT_CSV = f"{PROCESSED_FOLDER}/all_documents_extracted.csv"
PT_OT_CSV = f"{PROCESSED_FOLDER}/pt_ot_extracted.csv"
HOME_HEALTH_CSV = f"{PROCESSED_FOLDER}/home_health_extracted.csv"
CHECKPOINT_FILE = f"{PROCESSED_FOLDER}/extraction_checkpoint.csv"
STATS_FILE = f"{PROCESSED_FOLDER}/extraction_statistics.json"

# Processing settings
CHECKPOINT_INTERVAL = 50  # Save every 50 PDFs
OCR_DPI = 200
MIN_TEXT_LENGTH = 50
MAX_GARBAGE_RATIO = 0.3
OCR_MAX_RETRIES = 3

print("Configuration set!")
print(f"  Home Health Path: {HOME_HEALTH_PATH}")
print(f"  PT/OT Path: {PT_OT_PATH}")
print(f"  Output folder: {PROJECT_FOLDER}")

### 1.4 Create Project Directories

In [ ]:
# Create all necessary directories
directories = [
    PROJECT_FOLDER,
    DATA_FOLDER,
    PROCESSED_FOLDER,
    f"{DATA_FOLDER}/train_test_split",
    f"{PROJECT_FOLDER}/models",
    f"{PROJECT_FOLDER}/models/checkpoints",
    f"{PROJECT_FOLDER}/models/embeddings",
    f"{PROJECT_FOLDER}/results",
    f"{PROJECT_FOLDER}/results/visualizations"
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    
print(f"✓ Created {len(directories)} project directories")
for d in directories:
    print(f"  📁 {d.split('/')[-1]}")

### 1.5 Verify PDF Folders Exist

In [ ]:
# Verify PDF folders exist and count files
def count_pdfs(path):
    """Count PDF files in directory"""
    if not os.path.exists(path):
        return 0, False
    pdfs = list(Path(path).rglob('*.pdf'))
    return len(pdfs), True

pt_ot_count, pt_ot_exists = count_pdfs(PT_OT_PATH)
hh_count, hh_exists = count_pdfs(HOME_HEALTH_PATH)

print("PDF Folder Verification")
print("=" * 50)

if pt_ot_exists:
    print(f"✓ PT/OT folder found: {pt_ot_count} PDFs")
else:
    print(f"❌ PT/OT folder NOT found at: {PT_OT_PATH}")
    print("   Please check the path and update PT_OT_PATH above")

if hh_exists:
    print(f"✓ Home Health folder found: {hh_count} PDFs")
else:
    print(f"❌ Home Health folder NOT found at: {HOME_HEALTH_PATH}")
    print("   Please check the path and update HOME_HEALTH_PATH above")

total_pdfs = pt_ot_count + hh_count
print(f"\nTotal PDFs to process: {total_pdfs}")

if total_pdfs == 0:
    print("\n⚠️ WARNING: No PDFs found! Please verify your folder paths.")

---

## 🔧 Section 2: Core Extraction Classes

### 2.1 ICD-10 Code Validator

In [ ]:
class ICD10Validator:
    """
    Validates and extracts ICD-10-CM diagnosis codes from text.
    
    ICD-10-CM Format:
    - First character: Letter (A-Z except U)
    - Characters 2-3: Digits (00-99)
    - Optional: Decimal point + 1-4 alphanumeric characters
    
    Examples: I10, E11.9, M62.81, R26.81, Z91.81
    """
    
    def __init__(self):
        """Initialize with ICD-10 pattern"""
        self.pattern = re.compile(r'\b([A-TV-Z][0-9]{2})\.?([0-9A-Z]{1,4})?\b', re.IGNORECASE)
        
        # ICD-10-CM Chapter mappings
        self.chapters = {
            'A': 'Infectious diseases', 'B': 'Infectious diseases',
            'C': 'Neoplasms', 'D': 'Neoplasms/Blood diseases',
            'E': 'Endocrine/Metabolic', 'F': 'Mental disorders',
            'G': 'Nervous system', 'H': 'Eye/Ear',
            'I': 'Circulatory system', 'J': 'Respiratory system',
            'K': 'Digestive system', 'L': 'Skin diseases',
            'M': 'Musculoskeletal', 'N': 'Genitourinary',
            'O': 'Pregnancy', 'P': 'Perinatal conditions',
            'Q': 'Congenital malformations', 'R': 'Symptoms/Signs',
            'S': 'Injury', 'T': 'Injury/Poisoning',
            'V': 'External causes', 'W': 'External causes',
            'X': 'External causes', 'Y': 'External causes',
            'Z': 'Factors influencing health'
        }
        
        # False positives to exclude
        self.false_positives = {'A00', 'B00', 'C00', 'D00', 'T00', 'V00', 'W00', 'X00', 'Y00'}
    
    def validate_code(self, code: str) -> bool:
        """Validate a single ICD-10-CM code"""
        if not code or len(code) < 3:
            return False
        code = code.upper().strip()
        if code[0] == 'U':  # Reserved
            return False
        if not code[0].isalpha():
            return False
        base = code[:3].replace('.', '')
        if len(base) < 3 or not base[1:3].isdigit():
            return False
        if code[:3] in self.false_positives:
            return False
        return True
    
    def normalize_code(self, base: str, extension: str = '') -> str:
        """Normalize ICD-10 code to standard format"""
        code = base.upper()
        if extension:
            code = f"{code}.{extension.upper()}"
        return code
    
    def extract_codes(self, text: str) -> List[str]:
        """Extract all valid ICD-10 codes from text"""
        if not text:
            return []
        matches = self.pattern.findall(text.upper())
        codes = []
        seen = set()
        for base, extension in matches:
            code = self.normalize_code(base, extension)
            if code not in seen and self.validate_code(code):
                codes.append(code)
                seen.add(code)
        return codes
    
    def get_chapter(self, code: str) -> str:
        """Get the chapter name for a code"""
        if not code:
            return 'Unknown'
        return self.chapters.get(code[0].upper(), 'Unknown')


# Test the validator
validator = ICD10Validator()
test_text = "Patient has I10 hypertension, E11.9 type 2 diabetes, and R26.81 unsteadiness."
codes = validator.extract_codes(test_text)
print(f"Test extraction: {codes}")
print(f"✓ ICD10Validator ready!")

### 2.2 Hybrid PDF Extractor

In [ ]:
class HybridPDFExtractor:
    """
    Extracts text from PDFs using hybrid approach:
    1. First attempts native text extraction (fast)
    2. Falls back to OCR if native extraction fails
    """
    
    def __init__(self):
        self.icd_validator = ICD10Validator()
    
    def is_text_quality_acceptable(self, text: str) -> bool:
        """Check if extracted text quality is acceptable"""
        if not text or len(text) < MIN_TEXT_LENGTH:
            return False
        
        # Check garbage ratio
        valid_chars = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,;:!?\'\-\n\t/()')
        garbage_count = sum(1 for c in text if c not in valid_chars)
        if garbage_count / len(text) > MAX_GARBAGE_RATIO:
            return False
        
        # Check reasonable word count
        words = text.split()
        if len(words) < 10:
            return False
        
        return True
    
    def extract_text_native(self, pdf_path: str) -> Tuple[str, bool]:
        """Extract text using pdfplumber (native)"""
        try:
            text_parts = []
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text_parts.append(page_text)
            full_text = '\n'.join(text_parts).strip()
            return full_text, bool(full_text)
        except Exception as e:
            return "", False
    
    def extract_text_ocr(self, pdf_path: str) -> Tuple[str, bool]:
        """Extract text using OCR (Tesseract)"""
        for attempt in range(OCR_MAX_RETRIES):
            try:
                # Convert PDF to images
                images = convert_from_path(pdf_path, dpi=OCR_DPI, fmt='jpeg')
                
                # OCR each page
                text_parts = []
                for image in images:
                    page_text = pytesseract.image_to_string(image, lang='eng')
                    if page_text:
                        text_parts.append(page_text)
                    del image
                
                del images
                gc.collect()
                
                full_text = '\n'.join(text_parts).strip()
                return full_text, bool(full_text)
                
            except Exception as e:
                if attempt < OCR_MAX_RETRIES - 1:
                    time.sleep(2)
        
        return "", False
    
    def smart_extract(self, pdf_path: str) -> Dict:
        """Smart extraction: try native first, OCR fallback"""
        filename = os.path.basename(pdf_path)
        
        # Try native extraction first
        text, native_success = self.extract_text_native(pdf_path)
        extraction_method = 'native'
        
        # Check quality, fallback to OCR if needed
        if not native_success or not self.is_text_quality_acceptable(text):
            ocr_text, ocr_success = self.extract_text_ocr(pdf_path)
            if ocr_success and (not text or len(ocr_text) > len(text)):
                text = ocr_text
                extraction_method = 'ocr'
        
        # Extract ICD-10 codes
        icd_codes = self.icd_validator.extract_codes(text) if text else []
        
        # Determine document type
        doc_type = self._determine_document_type(text, filename)
        
        # Determine success
        success = bool(text) and len(text) >= MIN_TEXT_LENGTH
        
        return {
            'filename': filename,
            'filepath': str(pdf_path),
            'document_type': doc_type,
            'full_text': text,
            'text_length': len(text),
            'icd10_codes': json.dumps(icd_codes),  # Store as JSON string
            'num_codes': len(icd_codes),
            'extraction_method': extraction_method,
            'success': success
        }
    
    def _determine_document_type(self, text: str, filename: str) -> str:
        """Determine document type from content"""
        if not text:
            return 'Unknown'
        text_upper = text[:2000].upper()
        filename_upper = filename.upper()
        
        if 'G0179' in text_upper or 'G0180' in text_upper:
            return 'Home Health Certification'
        elif '485' in filename_upper or 'HOME HEALTH' in text_upper:
            return 'Home Health'
        elif 'PHYSICAL THERAPY' in text_upper or 'PT EVAL' in text_upper:
            return 'Physical Therapy'
        elif 'OCCUPATIONAL THERAPY' in text_upper or 'OT EVAL' in text_upper:
            return 'Occupational Therapy'
        elif 'SPEECH' in text_upper:
            return 'Speech Therapy'
        else:
            return 'Other'


# Test the extractor
extractor = HybridPDFExtractor()
print("✓ HybridPDFExtractor ready!")

---

## 📄 Section 3: PDF Extraction

### 3.1 Test with 10 PDFs First (RECOMMENDED!)

In [ ]:
# ⚠️ RUN THIS FIRST - Test with small batch before full extraction!

def test_extraction(n_samples=5):
    """Test extraction on a few PDFs from each folder"""
    print("🧪 TESTING EXTRACTION ON SAMPLE PDFS")
    print("=" * 50)
    
    results = []
    
    # Sample from PT/OT
    pt_ot_pdfs = list(Path(PT_OT_PATH).rglob('*.pdf'))[:n_samples]
    print(f"\nTesting {len(pt_ot_pdfs)} PT/OT PDFs...")
    for pdf_path in pt_ot_pdfs:
        result = extractor.smart_extract(str(pdf_path))
        results.append(result)
        print(f"  ✓ {result['filename'][:30]}... | {result['extraction_method']} | {result['num_codes']} codes")
    
    # Sample from Home Health
    hh_pdfs = list(Path(HOME_HEALTH_PATH).rglob('*.pdf'))[:n_samples]
    print(f"\nTesting {len(hh_pdfs)} Home Health PDFs...")
    for pdf_path in hh_pdfs:
        result = extractor.smart_extract(str(pdf_path))
        results.append(result)
        print(f"  ✓ {result['filename'][:30]}... | {result['extraction_method']} | {result['num_codes']} codes")
    
    # Summary
    df = pd.DataFrame(results)
    print("\n" + "=" * 50)
    print("TEST SUMMARY")
    print("=" * 50)
    print(f"Total tested: {len(df)}")
    print(f"Successful: {df['success'].sum()}")
    print(f"Native extraction: {(df['extraction_method'] == 'native').sum()}")
    print(f"OCR extraction: {(df['extraction_method'] == 'ocr').sum()}")
    print(f"Total ICD-10 codes: {df['num_codes'].sum()}")
    print(f"Avg codes per doc: {df['num_codes'].mean():.1f}")
    
    return df

# Run test
test_df = test_extraction(n_samples=5)

### 3.2 View Sample Extraction Results

In [ ]:
# View a sample extraction in detail
if len(test_df) > 0:
    sample = test_df.iloc[0]
    print("SAMPLE EXTRACTION RESULT")
    print("=" * 50)
    print(f"Filename: {sample['filename']}")
    print(f"Document Type: {sample['document_type']}")
    print(f"Extraction Method: {sample['extraction_method']}")
    print(f"Text Length: {sample['text_length']} characters")
    print(f"\nICD-10 Codes Found ({sample['num_codes']}):")
    codes = json.loads(sample['icd10_codes'])
    for code in codes[:10]:  # Show first 10
        chapter = validator.get_chapter(code)
        print(f"  • {code} - {chapter}")
    if len(codes) > 10:
        print(f"  ... and {len(codes) - 10} more")
    
    print(f"\nFirst 500 characters of text:")
    print("-" * 50)
    print(sample['full_text'][:500])

### 3.3 Full Extraction Function

In [ ]:
def process_pdf_directory(
    directory: str,
    output_csv: str,
    doc_type_label: str = "Unknown",
    checkpoint_file: str = None,
    resume: bool = True
) -> pd.DataFrame:
    """
    Process all PDFs in a directory with checkpointing.
    
    Args:
        directory: Path to PDF directory
        output_csv: Output CSV path
        doc_type_label: Label for this batch (e.g., 'PT/OT')
        checkpoint_file: Path for checkpoint saves
        resume: If True, resume from checkpoint
    
    Returns:
        DataFrame with extraction results
    """
    # Find all PDFs
    pdf_files = list(Path(directory).rglob('*.pdf'))
    print(f"\n{'='*60}")
    print(f"PROCESSING {doc_type_label.upper()} DOCUMENTS")
    print(f"{'='*60}")
    print(f"Found {len(pdf_files)} PDF files")
    
    if not pdf_files:
        print("⚠️ No PDF files found!")
        return pd.DataFrame()
    
    # Load checkpoint if exists
    processed_files = set()
    results = []
    
    if resume and checkpoint_file and os.path.exists(checkpoint_file):
        checkpoint_df = pd.read_csv(checkpoint_file)
        processed_files = set(checkpoint_df['filename'].tolist())
        results = checkpoint_df.to_dict('records')
        print(f"📂 Resuming from checkpoint: {len(processed_files)} already processed")
    
    # Filter remaining
    remaining = [f for f in pdf_files if os.path.basename(f) not in processed_files]
    print(f"Processing {len(remaining)} remaining files...")
    
    if len(remaining) == 0:
        print("✓ All files already processed!")
        return pd.DataFrame(results)
    
    # Process with progress bar
    start_time = time.time()
    batch_count = 0
    native_count = 0
    ocr_count = 0
    
    for pdf_path in tqdm(remaining, desc=f"Extracting {doc_type_label}"):
        try:
            result = extractor.smart_extract(str(pdf_path))
            results.append(result)
            
            if result['extraction_method'] == 'native':
                native_count += 1
            else:
                ocr_count += 1
                
        except Exception as e:
            results.append({
                'filename': os.path.basename(pdf_path),
                'filepath': str(pdf_path),
                'document_type': 'Unknown',
                'full_text': '',
                'text_length': 0,
                'icd10_codes': '[]',
                'num_codes': 0,
                'extraction_method': 'failed',
                'success': False
            })
        
        batch_count += 1
        
        # Save checkpoint
        if batch_count % CHECKPOINT_INTERVAL == 0:
            df = pd.DataFrame(results)
            if checkpoint_file:
                df.to_csv(checkpoint_file, index=False)
            elapsed = time.time() - start_time
            rate = batch_count / elapsed * 60
            remaining_time = (len(remaining) - batch_count) / rate if rate > 0 else 0
            print(f"\n  💾 Checkpoint: {len(results)} docs | {rate:.1f} docs/min | ~{remaining_time:.0f} min remaining")
            gc.collect()
    
    # Final save
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    
    # Statistics
    elapsed = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"{doc_type_label.upper()} EXTRACTION COMPLETE")
    print(f"{'='*60}")
    print(f"Total processed: {len(df)}")
    print(f"Successful: {df['success'].sum()}")
    print(f"Native extraction: {native_count}")
    print(f"OCR extraction: {ocr_count}")
    print(f"Time elapsed: {elapsed/60:.1f} minutes")
    print(f"Saved to: {output_csv}")
    
    return df

### 3.4 Process PT/OT Documents (346 files)

In [ ]:
# Process PT/OT documents
# Estimated time: ~45-60 minutes

pt_ot_df = process_pdf_directory(
    directory=PT_OT_PATH,
    output_csv=PT_OT_CSV,
    doc_type_label="PT/OT",
    checkpoint_file=f"{PROCESSED_FOLDER}/pt_ot_checkpoint.csv",
    resume=True
)

### 3.5 Process Home Health Documents (485 files)

In [ ]:
# Process Home Health documents
# Estimated time: ~60-90 minutes

hh_df = process_pdf_directory(
    directory=HOME_HEALTH_PATH,
    output_csv=HOME_HEALTH_CSV,
    doc_type_label="Home Health",
    checkpoint_file=f"{PROCESSED_FOLDER}/home_health_checkpoint.csv",
    resume=True
)

### 3.6 Combine All Results

In [ ]:
# Combine PT/OT and Home Health datasets
print("Combining datasets...")

# Load if not in memory
if 'pt_ot_df' not in dir() or pt_ot_df is None:
    if os.path.exists(PT_OT_CSV):
        pt_ot_df = pd.read_csv(PT_OT_CSV)
    else:
        pt_ot_df = pd.DataFrame()

if 'hh_df' not in dir() or hh_df is None:
    if os.path.exists(HOME_HEALTH_CSV):
        hh_df = pd.read_csv(HOME_HEALTH_CSV)
    else:
        hh_df = pd.DataFrame()

# Add source labels
if len(pt_ot_df) > 0:
    pt_ot_df['source'] = 'PT/OT'
if len(hh_df) > 0:
    hh_df['source'] = 'Home Health'

# Combine
combined_df = pd.concat([pt_ot_df, hh_df], ignore_index=True)
combined_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n{'='*60}")
print("COMBINED DATASET SAVED")
print(f"{'='*60}")
print(f"PT/OT documents: {len(pt_ot_df)}")
print(f"Home Health documents: {len(hh_df)}")
print(f"Total documents: {len(combined_df)}")
print(f"Saved to: {OUTPUT_CSV}")

---

## 📊 Section 4: Dataset Analysis

### 4.1 Overall Statistics

In [ ]:
# Load combined data if not in memory
if 'combined_df' not in dir():
    combined_df = pd.read_csv(OUTPUT_CSV)

print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total documents: {len(combined_df)}")
print(f"Successful extractions: {combined_df['success'].sum()} ({combined_df['success'].mean()*100:.1f}%)")
print(f"\nExtraction Methods:")
print(combined_df['extraction_method'].value_counts().to_string())
print(f"\nDocument Types:")
print(combined_df['document_type'].value_counts().to_string())
print(f"\nText Length Statistics:")
print(combined_df['text_length'].describe().to_string())

### 4.2 ICD-10 Code Analysis

In [ ]:
# Analyze ICD-10 codes
print("ICD-10 CODE ANALYSIS")
print("=" * 60)

# Collect all codes
all_codes = []
for codes_str in combined_df['icd10_codes']:
    try:
        codes = json.loads(codes_str) if isinstance(codes_str, str) else codes_str
        all_codes.extend(codes)
    except:
        pass

# Count frequencies
code_counts = Counter(all_codes)
unique_codes = len(code_counts)

print(f"Total ICD-10 codes found: {len(all_codes):,}")
print(f"Unique ICD-10 codes: {unique_codes}")
print(f"Codes per document (avg): {len(all_codes)/len(combined_df):.1f}")
print(f"\nCodes per document distribution:")
print(combined_df['num_codes'].describe().to_string())

# Top 20 codes
print(f"\nTop 20 Most Frequent ICD-10 Codes:")
print("-" * 60)
for i, (code, count) in enumerate(code_counts.most_common(20), 1):
    chapter = validator.get_chapter(code)
    pct = count / len(combined_df) * 100
    print(f"{i:2}. {code:10} | {count:4} docs ({pct:5.1f}%) | {chapter}")

### 4.3 Code Frequency Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Top 15 codes bar chart
top_15 = code_counts.most_common(15)
codes, counts = zip(*top_15)
ax1 = axes[0, 0]
bars = ax1.barh(range(len(codes)), counts, color='steelblue')
ax1.set_yticks(range(len(codes)))
ax1.set_yticklabels(codes)
ax1.invert_yaxis()
ax1.set_xlabel('Document Count')
ax1.set_title('Top 15 Most Frequent ICD-10 Codes', fontsize=12, fontweight='bold')

# 2. Extraction method pie chart
ax2 = axes[0, 1]
method_counts = combined_df['extraction_method'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c']
ax2.pie(method_counts.values, labels=method_counts.index, autopct='%1.1f%%', 
        colors=colors[:len(method_counts)], startangle=90)
ax2.set_title('Extraction Method Distribution', fontsize=12, fontweight='bold')

# 3. Codes per document histogram
ax3 = axes[1, 0]
ax3.hist(combined_df['num_codes'], bins=30, color='coral', edgecolor='black', alpha=0.7)
ax3.axvline(combined_df['num_codes'].mean(), color='red', linestyle='--', 
            label=f'Mean: {combined_df["num_codes"].mean():.1f}')
ax3.set_xlabel('Number of ICD-10 Codes')
ax3.set_ylabel('Number of Documents')
ax3.set_title('ICD-10 Codes per Document', fontsize=12, fontweight='bold')
ax3.legend()

# 4. Code frequency distribution (log scale)
ax4 = axes[1, 1]
freq_values = sorted(code_counts.values(), reverse=True)
ax4.plot(range(1, len(freq_values)+1), freq_values, 'b-', linewidth=1.5)
ax4.set_xlabel('Code Rank')
ax4.set_ylabel('Frequency (log scale)')
ax4.set_yscale('log')
ax4.set_title('Code Frequency Distribution (Zipf\'s Law)', fontsize=12, fontweight='bold')
ax4.axhline(y=5, color='r', linestyle='--', alpha=0.7, label='Min frequency = 5')
ax4.legend()

plt.tight_layout()
plt.savefig(f"{PROJECT_FOLDER}/results/visualizations/extraction_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Visualization saved to: {PROJECT_FOLDER}/results/visualizations/extraction_analysis.png")

### 4.4 Save Statistics

In [ ]:
# Calculate and save comprehensive statistics

# Frequency buckets
freq_100_plus = sum(1 for c, f in code_counts.items() if f >= 100)
freq_10_to_99 = sum(1 for c, f in code_counts.items() if 10 <= f < 100)
freq_5_to_9 = sum(1 for c, f in code_counts.items() if 5 <= f < 10)
freq_under_5 = sum(1 for c, f in code_counts.items() if f < 5)

statistics = {
    'extraction_date': time.strftime('%Y-%m-%d %H:%M:%S'),
    'total_documents': len(combined_df),
    'successful_extractions': int(combined_df['success'].sum()),
    'success_rate': float(combined_df['success'].mean()),
    'pt_ot_documents': int(len(pt_ot_df)) if 'pt_ot_df' in dir() else 0,
    'home_health_documents': int(len(hh_df)) if 'hh_df' in dir() else 0,
    'native_extractions': int((combined_df['extraction_method'] == 'native').sum()),
    'ocr_extractions': int((combined_df['extraction_method'] == 'ocr').sum()),
    'total_icd10_codes': len(all_codes),
    'unique_icd10_codes': unique_codes,
    'avg_codes_per_doc': float(combined_df['num_codes'].mean()),
    'median_codes_per_doc': float(combined_df['num_codes'].median()),
    'max_codes_per_doc': int(combined_df['num_codes'].max()),
    'min_codes_per_doc': int(combined_df['num_codes'].min()),
    'codes_freq_100_plus': freq_100_plus,
    'codes_freq_10_to_99': freq_10_to_99,
    'codes_freq_5_to_9': freq_5_to_9,
    'codes_freq_under_5': freq_under_5,
    'top_20_codes': [{'code': c, 'count': int(f)} for c, f in code_counts.most_common(20)],
    'avg_text_length': float(combined_df['text_length'].mean()),
    'median_text_length': float(combined_df['text_length'].median())
}

# Save statistics
with open(STATS_FILE, 'w') as f:
    json.dump(statistics, f, indent=2)

# Also save code frequencies
code_freq_df = pd.DataFrame([
    {'code': code, 'frequency': freq, 'chapter': validator.get_chapter(code)}
    for code, freq in code_counts.items()
]).sort_values('frequency', ascending=False)

code_freq_df.to_csv(f"{PROCESSED_FOLDER}/code_frequencies.csv", index=False)

print("STATISTICS SUMMARY")
print("=" * 60)
print(f"Total documents: {statistics['total_documents']}")
print(f"Success rate: {statistics['success_rate']*100:.1f}%")
print(f"Native: {statistics['native_extractions']} | OCR: {statistics['ocr_extractions']}")
print(f"\nICD-10 Codes:")
print(f"  Total: {statistics['total_icd10_codes']:,}")
print(f"  Unique: {statistics['unique_icd10_codes']}")
print(f"  Avg per doc: {statistics['avg_codes_per_doc']:.1f}")
print(f"\nCode Frequency Distribution:")
print(f"  ≥100 occurrences: {freq_100_plus} codes")
print(f"  10-99 occurrences: {freq_10_to_99} codes")
print(f"  5-9 occurrences: {freq_5_to_9} codes")
print(f"  <5 occurrences: {freq_under_5} codes (will be filtered)")
print(f"\n✓ Statistics saved to: {STATS_FILE}")
print(f"✓ Code frequencies saved to: {PROCESSED_FOLDER}/code_frequencies.csv")

---

## ✅ Section 5: Final Summary

In [ ]:
print("\n" + "="*60)
print("🎉 PHASE 1 COMPLETE: DATA EXTRACTION")
print("="*60)
print("\nFiles Created:")
print(f"  📄 {OUTPUT_CSV}")
print(f"  📄 {PT_OT_CSV}")
print(f"  📄 {HOME_HEALTH_CSV}")
print(f"  📄 {STATS_FILE}")
print(f"  📊 {PROJECT_FOLDER}/results/visualizations/extraction_analysis.png")
print(f"  📄 {PROCESSED_FOLDER}/code_frequencies.csv")

print("\nKey Metrics:")
print(f"  ✓ {len(combined_df)} documents processed")
print(f"  ✓ {statistics['success_rate']*100:.1f}% extraction success rate")
print(f"  ✓ {statistics['unique_icd10_codes']} unique ICD-10 codes identified")
print(f"  ✓ {statistics['total_icd10_codes']:,} total code occurrences")

print("\n📌 Next Steps:")
print("  1. Run Notebook 2: Text Preprocessing")
print("  2. Build vocabulary and encode documents")
print("  3. Prepare labels and train/val/test splits")
print("  4. Run Notebook 3: CNN Model Training")

print("\n" + "="*60)